In [9]:
import re
import textwrap
from collections import defaultdict
from pathlib import Path
from typing import Iterable
from typing import List, Optional, Sequence, Dict, Tuple

import matplotlib.pyplot as plt
import numpy as np

from power_benchmark.results.data.ResultData import ResultData
from power_benchmark.results.data.ScenarioData import ScenarioResult

# Load Experiment Results

In [10]:
output_path = Path("../../output/")

result_dn = ResultData.load_from_dir(output_path / "DN")
result_grd = ResultData.load_from_dir(output_path / "Greedy")
result_ppo = ResultData.load_from_dir(output_path / "PPO")
result_ppo_perf = ResultData.load_from_dir(output_path / "PPO Perf")
result_grd_perf = ResultData.load_from_dir(output_path / "Greedy Perf")

results = [result_dn, result_grd, result_ppo]
results_perf = [result_ppo_perf, result_grd_perf]

result_names = [r.name for r in results]

AGENT_ALIAS = {
    "GNN PPO Agent": "PPO",
    "GnnAgent": "PPO",
    "DN": "Do-N.",
    "DoNothing": "Do-N.",
    "Greedy": "Greedy",
}

# Plot Metric Scores

In [ ]:
def natural_key(value: str):
    """Split text into text/number chunks so 'run2' sorts before 'run10'."""
    return [
        int(chunk) if chunk.isdigit() else chunk.lower()
        for chunk in re.split(r"(\d+)", str(value))
    ]


def natural_sorted(values: Iterable[str]) -> List[str]:
    return sorted(values, key=natural_key)


def slugify(value: str) -> str:
    slug = re.sub(r"[^\w\-]+", "_", str(value).strip())
    return re.sub(r"_+", "_", slug).strip("_").lower() or "unnamed"


def wrap_label(name: str, width: int = 18) -> str:
    return "\n".join(textwrap.wrap(str(name), width=width)) or str(name)

def collect_category_metrics(results) -> Dict[str, Dict[str, Dict[str, float]]]:
    """category -> metric -> result_name -> mean score"""
    collected: Dict[str, Dict[str, Dict[str, float]]] = defaultdict(lambda: defaultdict(dict))
    for result in results:
        for category, metrics in result.category_metrics_average.items():
            for metric, score in metrics.items():
                collected[category][metric][result.name] = float(score)
    return collected

COLORS = plt.rcParams["axes.prop_cycle"].by_key()["color"]

def plot_category_metrics(
    category: str,
    metric_scores: Dict[str, Dict[str, float]],
    result_names: List[str],
    annotate: bool = True,
    out_dir: Optional[Path] = None,
    dpi: int = 150,
):
    metrics = natural_sorted(metric_scores.keys())
    n_metrics, n_results = len(metrics), len(result_names)

    x = np.arange(n_metrics, dtype=float)
    bar_width = min(0.8 / max(n_results, 1), 0.35)
    fig_width = max(8.0, 1.15 * n_metrics * max(n_results, 1) ** 0.5 + 2.0)

    fig, ax = plt.subplots(figsize=(fig_width, 6.0))

    for idx, name in enumerate(result_names):
        scores = [metric_scores[metric].get(name, np.nan) for metric in metrics]
        offset = (idx - (n_results - 1) / 2.0) * bar_width

        bars = ax.bar(
            x + offset, scores, width=bar_width, label=name,
            color=COLORS[idx % len(COLORS)],
            edgecolor="white", linewidth=0.6, zorder=3,
        )

        if annotate:
            rotate = n_metrics * n_results > 18
            for bar, score in zip(bars, scores):
                if score is None or np.isnan(score):
                    continue
                ax.annotate(
                    f"{score:.1f}",
                    xy=(bar.get_x() + bar.get_width() / 2.0, bar.get_height()),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=7,
                    rotation=90 if rotate else 0, zorder=4,
                )

    ax.set_title(f"Average metric scores – category '{category}'", fontsize=13, pad=12)
    ax.set_xlabel("Metric")
    ax.set_ylabel("Average score")
    ax.set_xticks(x)
    ax.set_xticklabels([wrap_label(m) for m in metrics], fontsize=8)
    ax.set_ylim(0, 105)
    ax.set_axisbelow(True)
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)

    if n_results > 1:
        ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18),
                  ncol=min(n_results, 4), frameon=False, fontsize=9)

    fig.tight_layout()

    if out_dir is not None:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        path = out_dir / f"{category}.png"
        fig.savefig(path, dpi=dpi, bbox_inches="tight")
        print(f"saved -> {path}")

    plt.show()
    return fig, ax

In [ ]:
category_metrics = collect_category_metrics(results)

for category in natural_sorted(category_metrics):
    plot_category_metrics(
        category=category,
        metric_scores=category_metrics[category],
        result_names=result_names,
        out_dir=None
    )

# Compare single Scenarios

In [ ]:
def category_score(category) -> Optional[float]:
    """Score of one metric category: use a ready-made field if it exists,
    otherwise average the metric scores inside the category."""
    for attr in ("score", "average_score", "avg_score", "mean_score"):
        value = getattr(category, attr, None)
        if isinstance(value, (int, float)):
            return float(value)

    scores: List[float] = []
    for metric in getattr(category, "metrics", []):
        score_dict = metric.score_dict()
        scores.extend(s for s in score_dict.values() if s is not None)

    return float(np.mean(scores)) if scores else None


def run_label(single_result, idx: int) -> str:
    """Human readable run identifier with fallback to the positional index."""
    for attr in ("run_id", "run", "index", "id", "name"):
        value = getattr(single_result, attr, None)
        if value not in (None, ""):
            return str(value)
    return f"run{idx + 1}"


def collect_scenario_category_scores(results, scenario_name: str):
    """
    Returns:
        data:  category -> run_label -> result_name -> score
        runs:  naturally sorted run labels that carry at least one score
    """
    data: Dict[str, Dict[str, Dict[str, float]]] = defaultdict(lambda: defaultdict(dict))
    runs: set = set()

    for result in results:
        scenario = next(
            (s for s in result.scenarios if s.name == scenario_name), None
        )
        if scenario is None:
            print(f"[warn] '{result.name}' has no scenario '{scenario_name}'")
            continue

        for idx, single in enumerate(scenario.results):
            label = run_label(single, idx)
            for category in getattr(single, "metric_categories", []):
                score = category_score(category)
                if score is not None:
                    data[category.name][label][result.name] = score
                    runs.add(label)

    # drop runs without any score (keeps the x-axis free of empty slots)
    populated = {
        run for run in runs
        if any(run in per_run and per_run[run] for per_run in data.values())
    }
    return data, natural_sorted(populated)

AGENT_HATCHES = ["", "/", "o", "xxx", "\\\\\\", "+++"]

def plot_scenario_category_comparison(
    results,
    scenario_name: str,
    annotate: bool = True,
    ncols: int = 3,
    sharey: bool = True,
    out_dir: Optional[Path] = None,
    dpi: int = 150,
):
    """One subplot per category; x = run indexes, colour = category, hatch = agent."""
    data, runs = collect_scenario_category_scores(results, scenario_name)
    if not data or not runs:
        print(f"[warn] no category scores for scenario '{scenario_name}'")
        return None, None

    categories = natural_sorted(data.keys())
    result_names = [r.name for r in results]

    n_cat, n_res, n_run = len(categories), len(result_names), len(runs)
    ncols = min(ncols, n_cat)
    nrows = int(np.ceil(n_cat / ncols))

    x = np.arange(n_run, dtype=float)
    bar_width = min(0.8 / max(n_res, 1), 0.28)

    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(max(3.6 * ncols, 5.0), 3.4 * nrows + 1.2),
        sharey=sharey, squeeze=False,
    )
    flat_axes = axes.flatten()

    for c_idx, (ax, category) in enumerate(zip(flat_axes, categories)):
        colour = COLORS[c_idx % len(COLORS)]

        for r_idx, name in enumerate(result_names):
            scores = [data[category].get(run, {}).get(name, np.nan) for run in runs]
            offset = (r_idx - (n_res - 1) / 2.0) * bar_width

            bars = ax.bar(
                x + offset, scores, width=bar_width,
                color=colour,
                hatch=AGENT_HATCHES[r_idx % len(AGENT_HATCHES)],
                edgecolor="white", linewidth=0.7, zorder=3,
            )

            if annotate:
                for bar, score in zip(bars, scores):
                    if score is None or np.isnan(score):
                        continue
                    ax.annotate(
                        f"{score:.0f}",
                        xy=(bar.get_x() + bar.get_width() / 2.0, bar.get_height()),
                        xytext=(0, 2), textcoords="offset points",
                        ha="center", va="bottom", fontsize=7,
                        rotation=90 if n_res * n_run > 9 else 0, zorder=4,
                    )

        ax.set_title(wrap_label(category, 26), fontsize=10, color=colour)
        ax.set_xlabel("Run")
        ax.set_xticks(x)
        ax.set_xticklabels(runs, fontsize=8)
        ax.set_xlim(-0.5, n_run - 0.5)
        ax.set_ylim(0, 105)
        ax.set_axisbelow(True)
        ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)

    for row in range(nrows):
        axes[row][0].set_ylabel("Category score")

    for ax in flat_axes[n_cat:]:
        ax.set_visible(False)

    # legend encodes the agents via hatch only (colour is category specific)
    agent_handles = [
        plt.Rectangle((0, 0), 1, 1, facecolor="0.8",
                      hatch=AGENT_HATCHES[i % len(AGENT_HATCHES)],
                      edgecolor="white", linewidth=0.7)
        for i in range(n_res)
    ]
    fig.legend(
        agent_handles, result_names,
        loc="lower center", bbox_to_anchor=(0.5, -0.01),
        ncol=min(n_res, 4), frameon=False, fontsize=9, title="Agent",
    )

    fig.suptitle(f"Category scores per run – scenario '{scenario_name}'",
                 fontsize=13, y=0.995)
    fig.tight_layout(rect=(0, 0.06, 1, 0.97))

    if out_dir is not None:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        path = out_dir / f"{scenario_name}.png"
        fig.savefig(path, dpi=dpi, bbox_inches="tight")
        print(f"saved -> {path}")

    plt.show()
    return fig, axes

def plot_scenario_category_comparison_single_axes(
    results,
    scenario_name: str,
    annotate: bool = False,
    out_dir: Optional[Path] = None,
    dpi: int = 150,
):
    """Single axes; x = run indexes, colour = category, hatch = agent."""
    data, runs = collect_scenario_category_scores(results, scenario_name)
    if not data or not runs:
        print(f"[warn] no category scores for scenario '{scenario_name}'")
        return None, None

    categories = natural_sorted(data.keys())
    result_names = [r.name for r in results]

    n_cat, n_res, n_run = len(categories), len(result_names), len(runs)
    n_bars = n_cat * n_res

    bar_width = 0.82 / n_bars
    x = np.arange(n_run, dtype=float)

    # Mindestbreite, damit zwei Legenden nebeneinander Platz haben
    fig_width = max(9.0, 1.15 * n_run * n_bars ** 0.5 + 2.0)
    if n_run <= 2:
        fig_width = max(fig_width, 11.0)

    fig, ax = plt.subplots(figsize=(fig_width, 6.8))

    for c_idx, category in enumerate(categories):
        colour = COLORS[c_idx % len(COLORS)]
        for r_idx, name in enumerate(result_names):
            slot = c_idx * n_res + r_idx
            offset = (slot - (n_bars - 1) / 2.0) * bar_width
            scores = [data[category].get(run, {}).get(name, np.nan) for run in runs]

            bars = ax.bar(
                x + offset, scores, width=bar_width,
                color=colour,
                hatch=AGENT_HATCHES[r_idx % len(AGENT_HATCHES)],
                edgecolor="white", linewidth=0.7, zorder=3,
            )

            if annotate:
                for bar, score in zip(bars, scores):
                    if score is None or np.isnan(score):
                        continue
                    ax.annotate(
                        f"{score:.0f}",
                        xy=(bar.get_x() + bar.get_width() / 2.0, bar.get_height()),
                        xytext=(0, 2), textcoords="offset points",
                        ha="center", va="bottom", fontsize=6.5,
                        rotation=90, zorder=4,
                    )

    for pos in x[:-1]:
        ax.axvline(pos + 0.5, color="0.85", linewidth=0.8, zorder=0)

    ax.set_title(f"Category scores per run – scenario '{scenario_name}'", fontsize=13, pad=12)
    ax.set_xlabel("Run")
    ax.set_ylabel("Category score")
    ax.set_xticks(x)
    ax.set_xticklabels(runs, fontsize=9)
    ax.set_xlim(-0.5, n_run - 0.5)
    ax.set_ylim(0, 105)
    ax.set_axisbelow(True)
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)

    # colour legend = categories
    cat_handles = [
        plt.Rectangle((0, 0), 1, 1, facecolor=COLORS[i % len(COLORS)], edgecolor="white")
        for i in range(n_cat)
    ]

    # hatch legend = agents (größere Boxen, dichteres Muster, dunkle Kante)
    agent_handles = [
        plt.Rectangle((0, 0), 1, 1, facecolor="0.85",
                      hatch=AGENT_HATCHES[i % len(AGENT_HATCHES)] * 2,
                      edgecolor="0.25", linewidth=0.8)
        for i in range(n_res)
    ]

    # Platz unten für die Legenden reservieren
    fig.tight_layout(rect=(0, 0.1, 1, 1))

    # Bei schmaler Figur Legenden untereinander stapeln, sonst nebeneinander
    stack = fig.get_size_inches()[0] < 11.0

    if stack:
        cat_anchor = (0.5, 0.095)
        agent_anchor = (0.5, 0.048)
    else:
        cat_anchor = (0.30, 0.095)
        agent_anchor = (0.72, 0.095)

    fig.legend(
        cat_handles, categories,
        loc="upper center", bbox_to_anchor=cat_anchor,
        bbox_transform=fig.transFigure,
        ncol=min(n_cat, 3), frameon=False, fontsize=9, title="Category",
        handlelength=2.0, handleheight=1.4, handletextpad=0.7,
    )

    fig.legend(
        agent_handles, result_names,
        loc="upper center", bbox_to_anchor=agent_anchor,
        bbox_transform=fig.transFigure,
        ncol=min(n_res, 3), frameon=False, fontsize=9, title="Agent",
        handlelength=3.0, handleheight=2.0, handletextpad=0.8,
        labelspacing=0.8, columnspacing=1.6,
    )

    if out_dir is not None:
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        path = out_dir / f"{scenario_name}.png"
        fig.savefig(path, dpi=dpi, bbox_inches="tight")
        print(f"saved -> {path}")

    plt.show()
    return fig, ax

In [ ]:
scenario_names = natural_sorted({
    s.name for r in results for s in r.scenarios
})

# all scenarios
for name in scenario_names:
    plot_scenario_category_comparison_single_axes(results, name, out_dir=None)

# Total Score Comparison

In [ ]:
def get_scenario_scores(
    results: Sequence["ResultData"],
    scenario_names: Sequence[str],
    substring_match: bool = False,
) -> Dict[str, Dict[str, Optional[float]]]:
    """agent_name -> {scenario_name -> average_score (or None if missing)}"""
    table: Dict[str, Dict[str, Optional[float]]] = {}

    for result in results:
        by_name = {s.name: s for s in result.scenarios}
        row: Dict[str, Optional[float]] = {}

        for wanted in scenario_names:
            if wanted in by_name:
                row[wanted] = by_name[wanted].average_score
            elif substring_match:
                matches = [s for n, s in by_name.items() if wanted in n]
                if matches:
                    row[wanted] = float(np.mean([m.average_score for m in matches]))
                else:
                    row[wanted] = None
            else:
                row[wanted] = None

        table[result.name] = row

    return table


def plot_scenario_scores(
    results: Sequence["ResultData"],
    scenario_names: Sequence[str],
    substring_match: bool = False,
    title: str = "Average score per scenario",
    figsize=(11, 5),
    annotate: bool = True,
    show_means: bool = True,
    mean_over_all_scenarios: bool = False,  # True -> result.average_score (alle Szenarien)
    save_path: Optional[str] = None,
):
    table = get_scenario_scores(results, scenario_names, substring_match)

    agents = list(table.keys())
    n_agents = len(agents)
    n_scen = len(scenario_names)

    x = np.arange(n_scen)
    width = 0.8 / n_agents

    fig, ax = plt.subplots(figsize=figsize)

    means: Dict[str, Optional[float]] = {}

    for i, (agent, result) in enumerate(zip(agents, results)):
        offset = (i - (n_agents - 1) / 2) * width
        scores = [table[agent][s] for s in scenario_names]
        plot_scores = [0.0 if s is None else s for s in scores]

        bars = ax.bar(x + offset, plot_scores, width, label=agent)
        color = bars[0].get_facecolor()

        if annotate:
            for b, s in zip(bars, scores):
                label = "n/a" if s is None else f"{s:.1f}"
                ax.annotate(
                    label,
                    (b.get_x() + b.get_width() / 2, b.get_height()),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8, rotation=90,
                )

        # --- Mittelwert-Linie ---
        valid = [s for s in scores if s is not None]
        mean_val = (
            result.average_score if mean_over_all_scenarios
            else (float(np.mean(valid)) if valid else None)
        )
        means[agent] = mean_val

        if show_means and mean_val is not None:
            ax.axhline(
                mean_val, color=color, linestyle="--", linewidth=1.5,
                alpha=0.9, zorder=3,
            )
            ax.annotate(
                f"Ø {AGENT_ALIAS[agent]}\n {mean_val:.1f}",
                xy=(1.0, mean_val), xycoords=("axes fraction", "data"),
                xytext=(4, 0), textcoords="offset points",
                ha="left", va="center", fontsize=8, color=color, clip_on=False,
            )

    ax.set_xticks(x)
    ax.set_xticklabels(scenario_names, rotation=30, ha="right")
    ax.set_ylabel("Average score")
    ax.set_title(title)
    ylim = (0,100)
    ax.set_ylim(*ylim)
    ax.set_yticks(np.arange(0, 101, 10))
    ax.grid(axis="y", alpha=0.3)

    # Legende unterhalb (Balken + gestrichelte Linie erklären)
    handles, labels = ax.get_legend_handles_labels()
    if show_means:
        from matplotlib.lines import Line2D
        handles.append(Line2D([0], [0], color="gray", linestyle="--", linewidth=1.5))
        labels.append("Mean (per agent)")

    ax.legend(
        handles, labels, title="Agent",
        loc="upper center", bbox_to_anchor=(0.5, -0.28),
        ncol=min(len(labels), 4), frameon=False,
    )

    fig.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=200, bbox_inches="tight")

    return fig, ax, table, means

In [ ]:
scenarios = ["14B_UP_GN0", "14B_UP_LD0", "30B_UP_LD5", "30B_UP_LD11", "30B_MT_LN", "30B_MT_GN", "30B_RC_LN"]

fig, ax, table, means = plot_scenario_scores(results, scenarios, title="Average score in the diverse scenario experiment")

In [ ]:
scenarios = ["14B_UP_GN0-", "14B_UP_GN0", "14B_UP_GN0+"]
fig, ax, table, means = plot_scenario_scores(results, scenarios, title="Average score in the generator stress test scenario experiment")

scenarios = ["14B_UP_LD0-", "14B_UP_LD0", "14B_UP_LD0+"]
fig, ax, table, means = plot_scenario_scores(results, scenarios, title="Average score in the load stress test scenario experiment")

scenarios = ["30B_RC_LN-", "30B_RC_LN", "30B_RC_LN+"]
fig, ax, table, means = plot_scenario_scores(results, scenarios, title="Average score in the line capacity stress test scenario experiment")

# Cost vs Grid Stability Correlation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

STABILITY_CATEGORIES = ["overload_mitigation", "voltage_violation_mitigation", "survival"]

def get_cost_stability(result):
    costs, stability = [], []
    for scenario in result.scenarios:
        cat_scores = scenario.average_category_scores
        if "costs" not in cat_scores:
            continue
        stab_vals = [cat_scores[c] for c in STABILITY_CATEGORIES if c in cat_scores]
        if not stab_vals:
            continue
        costs.append(cat_scores["costs"])
        stability.append(np.mean(stab_vals))
    return np.array(costs), np.array(stability)

def agent_label(result):
    return AGENT_ALIAS.get(result.agent, result.agent)

# --- Per-agent subplots ---
fig, axes = plt.subplots(1, len(results), figsize=(5 * len(results), 5), sharex=True, sharey=True)

all_costs, all_stability = [], []

for ax, result in zip(axes, results):
    costs, stability = get_cost_stability(result)
    all_costs.append(costs)
    all_stability.append(stability)

    ax.scatter(costs, stability, alpha=0.7)

    if len(costs) > 1:
        r, p = pearsonr(costs, stability)
        m, b = np.polyfit(costs, stability, 1)
        x_line = np.linspace(costs.min(), costs.max(), 100)
        ax.plot(x_line, m * x_line + b, color="red", linestyle="--")
        ax.set_title(f"{agent_label(result)}\nr={r:.2f}, p={p:.3f}")
    else:
        ax.set_title(agent_label(result))

    ax.set_xlabel("Cost score")
    ax.set_ylabel("Grid stability score (avg)")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# --- Combined plot across all agents ---
all_costs = np.concatenate(all_costs)
all_stability = np.concatenate(all_stability)

fig, ax = plt.subplots(figsize=(6, 5))
for result in results:
    costs, stability = get_cost_stability(result)
    ax.scatter(costs, stability, alpha=0.7, label=agent_label(result))

if len(all_costs) > 1:
    r, p = pearsonr(all_costs, all_stability)
    m, b = np.polyfit(all_costs, all_stability, 1)
    x_line = np.linspace(all_costs.min(), all_costs.max(), 100)
    ax.plot(x_line, m * x_line + b, color="black", linestyle="--", label="Overall fit")
    ax.set_title(f"Costs vs. Grid Stability (all agents)\nr={r:.2f}, p={p:.3f}")

ax.set_xlabel("Cost score")
ax.set_ylabel("Grid stability score (avg)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

ACTION_METRICS = ["timesteps_without_actions", "substation_changes"]
OUTCOME_METRICS = ["active_power_loss_timestep_percent", "timesteps_with_load_shedding", "load_shedding_timestep_percent"]
STABILITY_CATEGORIES = ["overload_mitigation", "voltage_violation_mitigation", "survival"]

def subset_score(single_result, category_name, metric_names_subset):
    """Weighted average score of a subset of metrics within one category, for one SingleResult."""
    for category in single_result.metric_categories:
        if category.name != category_name:
            continue
        scores = []
        for metric in category.metrics:
            if metric.name in metric_names_subset:
                scores += [(s, w) for s, w in metric.score_weight() if s is not None]
        weight_sum = sum(w for _, w in scores)
        if weight_sum == 0:
            return None
        return sum(s * w for s, w in scores) / weight_sum
    return None

def scenario_subset_score(scenario, category_name, metric_names_subset):
    """Average of subset_score across all non-failed results in a scenario."""
    vals = [
        subset_score(r, category_name, metric_names_subset)
        for r in scenario.non_failed_results
    ]
    vals = [v for v in vals if v is not None]
    return float(np.mean(vals)) if vals else None

def get_cost_split_and_stability(result):
    action_scores, outcome_scores, stability_scores = [], [], []
    for scenario in result.scenarios:
        cat_scores = scenario.average_category_scores
        stab_vals = [cat_scores[c] for c in STABILITY_CATEGORIES if c in cat_scores]
        if not stab_vals:
            continue

        action = scenario_subset_score(scenario, "costs", ACTION_METRICS)
        outcome = scenario_subset_score(scenario, "costs", OUTCOME_METRICS)

        if action is None or outcome is None:
            continue

        action_scores.append(action)
        outcome_scores.append(outcome)
        stability_scores.append(np.mean(stab_vals))

    return np.array(action_scores), np.array(outcome_scores), np.array(stability_scores)

def agent_label(result):
    return AGENT_ALIAS.get(result.agent, result.agent)

# --- Plot: Action-cost vs Stability, and Outcome-cost vs Stability ---
fig, axes = plt.subplots(2, len(results), figsize=(5 * len(results), 9), sharex="row", sharey=True)

for col, result in enumerate(results):
    action, outcome, stability = get_cost_split_and_stability(result)

    for row, (scores, title) in enumerate([(action, "Action-cost score (no-action / switching)"),
                                            (outcome, "Outcome-cost score (shedding / loss)")]):
        ax = axes[row, col]
        ax.scatter(scores, stability, alpha=0.7)

        if len(scores) > 1:
            r, p = pearsonr(scores, stability)
            m, b = np.polyfit(scores, stability, 1)
            x_line = np.linspace(scores.min(), scores.max(), 100)
            ax.plot(x_line, m * x_line + b, color="red", linestyle="--")
            ax.set_title(f"{agent_label(result)}\nr={r:.2f}, p={p:.3f}")

        ax.set_xlabel(title)
        if col == 0:
            ax.set_ylabel("Grid stability score (avg)")
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Grid Size

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

def grid_size_of(scenario_name: str) -> str:
    if scenario_name.startswith("14B"):
        return "14-bus"
    elif scenario_name.startswith("30B"):
        return "30-bus"
    else:
        return "other"

def build_grid_size_df(results) -> pd.DataFrame:
    rows = []
    for result in results:
        agent = agent_label(result)
        for scenario in result.scenarios:
            size = grid_size_of(scenario.name)
            if size == "other":
                continue

            row = {
                "agent": agent,
                "scenario": scenario.name,
                "grid_size": size,
                "total_score": scenario.average_score,
                **scenario.average_category_scores,
            }
            rows.append(row)
    return pd.DataFrame(rows)

df = build_grid_size_df(results)
df.head()

In [ ]:
categories_to_plot = ["costs"] + STABILITY_CATEGORIES

fig, axes = plt.subplots(1, len(categories_to_plot), figsize=(5 * len(categories_to_plot), 5), sharey=True)

for ax, cat in zip(axes, categories_to_plot):
    pivot = df.groupby(["agent", "grid_size"])[cat].mean().unstack()
    pivot.plot(kind="bar", ax=ax, rot=0)
    ax.set_title(cat)
    ax.set_ylabel("Score" if ax is axes[0] else "")
    ax.grid(alpha=0.3, axis="y")
    ax.legend(title="Grid size")

plt.tight_layout()
plt.show()

# LaTeX Table Generation

## Helpers

In [11]:
def _format_value(value: float, bold: bool = False) -> str:
    s = f"{value:.2f}"
    return f"\\textbf{{{s}}}" if bold else s


def _build_latex_table(
    rows: List[Tuple[str, List[float]]],
    caption: str,
    label: str,
    separate_first_row: bool = True,
) -> str:
    body_lines = []
    first_row_included = False

    for i, (row_label, values) in enumerate(rows):
        is_first_row = i == 0

        if not is_first_row and all(v == 0 for v in values):
            continue

        best_score = max(values)
        formatted = [
            _format_value(v, bold=(v == best_score))
            for v in values
        ]
        body_lines.append(f"    {row_label} & " + " & ".join(formatted) + r" \\")

        if separate_first_row and is_first_row:
            body_lines.append(r"    \midrule")
            first_row_included = True

    body = "\n".join(body_lines)

    return (
        f"\\agentresulttable\n"
        f"  {{{caption}}}\n"
        f"  {{{label}}}\n"
        f"  {{\n{body}\n  }}"
    )


def _avg_score_for_scenarios(scenarios: List[ScenarioResult]) -> float:
    return (
        sum(s.average_score for s in scenarios) / len(scenarios)
        if scenarios else 0.0
    )


def _avg_category_scores_for_scenarios(scenarios: List[ScenarioResult]) -> Dict[str, float]:
    gathered: Dict[str, List[float]] = defaultdict(list)
    for s in scenarios:
        for key, score in s.average_category_scores.items():
            gathered[key].append(score)
    return {k: float(np.mean(v)) for k, v in gathered.items()}

def _format_name(name: str) -> str:
    return " ".join(word.capitalize() for word in name.split("_"))

## Total Overview Table

In [12]:
def make_agent_comparison_table(
    results: List["ResultData"],
    caption: str = "Comparison of the agents using the average score and category scores",
    label: str = "tab:agent_comparison",
) -> str:

    category_names = list(results[0].average_category_scores.keys())

    rows = [("Average Score", [r.average_score for r in results])]
    for cat in category_names:
        rows.append((_format_name(cat), [r.average_category_scores.get(cat, 0.0) for r in results]))

    return _build_latex_table(rows, caption, label)

print(make_agent_comparison_table(results))

\agentresulttable
  {Comparison of the agents using the average score and category scores}
  {tab:agent_comparison}
  {
    Average Score & 58.20 & 41.25 & \textbf{64.86} \\
    \midrule
    Overload Mitigation & 53.90 & 51.44 & \textbf{73.22} \\
    Voltage Violation Mitigation & 41.54 & 30.21 & \textbf{56.85} \\
    Survival & 70.60 & 41.44 & \textbf{71.87} \\
    Costs & \textbf{62.89} & 42.34 & 43.09 \\
  }


## Specific Experiment Group Table

In [14]:
def make_scenario_comparison_table(
    results: List["ResultData"],
    experiment: str,
    scenario_names: List[str],
    caption: Optional[str] = None,
    label: Optional[str] = None,
) -> str:
    if caption is None:
        caption = f"Comparison for the {experiment} experiment"

    selected_per_result = [
        [s for s in r.scenarios if s.name in scenario_names]
        for r in results
    ]

    for r, sel in zip(results, selected_per_result):
        missing = set(scenario_names) - {s.name for s in sel}
        if missing:
            print(f"Warning: Agent '{r.name}': scenarios not found: {missing}")

    avg_scores = [_avg_score_for_scenarios(sel) for sel in selected_per_result]
    cat_scores_per_result = [_avg_category_scores_for_scenarios(sel) for sel in selected_per_result]

    category_names = []
    for cat_scores in cat_scores_per_result:
        for c in cat_scores:
            if c not in category_names:
                category_names.append(c)

    rows = [("Average Score", avg_scores)]
    for cat in category_names:
        rows.append((_format_name(cat), [cs.get(cat, 0.0) for cs in cat_scores_per_result]))

    if label is None:
        label = f"tab:scenario_comparison"

    return _build_latex_table(rows, caption, label)

### Standard Test

In [111]:
scenario_subset = ["14B_ORIG", "30B_ORIG"]

print(make_scenario_comparison_table(
    results,
    experiment="standard test",
    scenario_names=scenario_subset,
    label="tab:exp_overview_standard",
))

\agentresulttable
  {Comparison for standard test experiment}
  {tab:exp_overview_standard}
  {
    Average Score & 60.49 & 41.75 & \textbf{65.00} \\
    \midrule
    Overload Mitigation & 62.17 & 60.66 & \textbf{80.93} \\
    Voltage Violation Mitigation & 38.57 & 31.32 & \textbf{58.06} \\
    Survival & \textbf{64.76} & 30.21 & 63.71 \\
    Costs & \textbf{88.17} & 59.42 & 50.87 \\
  }


### Diverse Scenario Test

In [112]:
scenario_subset = ["14B_UP_GN0", "14B_UP_LD0", "30B_UP_LD5", "30B_UP_LD11", "30B_MT_LN", "30B_MT_GN", "30B_RC_LN"]

print(make_scenario_comparison_table(
    results,
    experiment="diverse scenarios",
    scenario_names=scenario_subset,
    label="tab:exp_overview_diverse",
))

scenario_subset = ["14B_UP_GN0"]

print(make_scenario_comparison_table(
    results,
    experiment="upscaled generator",
    scenario_names=scenario_subset,
    label="tab:exp_overview_gen",
))


scenario_subset = ["14B_UP_LD0", "30B_UP_LD5", "30B_UP_LD11"]

print(make_scenario_comparison_table(
    results,
    experiment="upscaled load",
    scenario_names=scenario_subset,
    label="tab:exp_overview_load",
))

scenario_subset = ["30B_MT_LN", "30B_MT_GN"]

print(make_scenario_comparison_table(
    results,
    experiment="maintenance",
    scenario_names=scenario_subset,
    label="tab:exp_overview_maintenance",
))

scenario_subset = ["30B_RC_LN"]

print(make_scenario_comparison_table(
    results,
    experiment="reduced line capacity",
    scenario_names=scenario_subset,
    label="tab:exp_overview_line_cap",
))

\agentresulttable
  {Comparison for diverse scenarios experiment}
  {tab:exp_overview_diverse}
  {
    Average Score & 61.25 & 45.25 & \textbf{76.14} \\
    \midrule
    Overload Mitigation & 44.18 & 47.65 & \textbf{79.40} \\
    Voltage Violation Mitigation & 40.91 & 40.09 & \textbf{71.69} \\
    Survival & 81.94 & 43.11 & \textbf{88.59} \\
    Costs & \textbf{73.99} & 57.18 & 41.14 \\
  }
\agentresulttable
  {Comparison for upscaled generator experiment}
  {tab:exp_overview_gen}
  {
    Average Score & 42.19 & 29.02 & \textbf{75.42} \\
    \midrule
    Overload Mitigation & 76.77 & 74.39 & \textbf{96.51} \\
    Voltage Violation Mitigation & 17.41 & 20.75 & \textbf{39.00} \\
    Survival & 30.21 & 4.17 & \textbf{100.00} \\
    Costs & \textbf{58.53} & 29.37 & 32.38 \\
  }
\agentresulttable
  {Comparison for upscaled load experiment}
  {tab:exp_overview_load}
  {
    Average Score & 58.38 & 47.96 & \textbf{65.44} \\
    \midrule
    Overload Mitigation & 39.55 & 46.48 & \textbf{68.97}

## Stress Test

In [114]:
scenario_subset = ["14B_UP_GN0-", "14B_UP_GN0", "14B_UP_GN0+"]

print(make_scenario_comparison_table(
    results,
    experiment="standard test",
    scenario_names=scenario_subset,
    label="tab:exp_overview_standard",
))

scenario_subset = ["14B_UP_LD0-", "14B_UP_LD0", "14B_UP_LD0+"]

print(make_scenario_comparison_table(
    results,
    experiment="standard test",
    scenario_names=scenario_subset,
    label="tab:exp_overview_standard",
))

scenario_subset = ["30B_RC_LN-", "30B_RC_LN", "30B_RC_LN+"]

print(make_scenario_comparison_table(
    results,
    experiment="standard test",
    scenario_names=scenario_subset,
    label="tab:exp_overview_standard",
))

\agentresulttable
  {Comparison for standard test experiment}
  {tab:exp_overview_standard}
  {
    Average Score & 33.39 & 20.74 & \textbf{40.21} \\
    \midrule
    Overload Mitigation & 54.35 & 51.26 & \textbf{59.02} \\
    Voltage Violation Mitigation & 13.19 & 14.36 & \textbf{21.23} \\
    Survival & 28.36 & 3.36 & \textbf{43.17} \\
    Costs & \textbf{46.99} & 24.64 & 31.65 \\
  }
\agentresulttable
  {Comparison for standard test experiment}
  {tab:exp_overview_standard}
  {
    Average Score & \textbf{45.78} & 30.67 & 41.19 \\
    \midrule
    Overload Mitigation & 65.22 & 52.90 & \textbf{65.36} \\
    Voltage Violation Mitigation & \textbf{13.72} & 11.51 & 12.69 \\
    Survival & \textbf{48.15} & 26.74 & 40.39 \\
    Costs & \textbf{63.89} & 36.31 & 52.21 \\
  }
\agentresulttable
  {Comparison for standard test experiment}
  {tab:exp_overview_standard}
  {
    Average Score & 72.29 & 46.89 & \textbf{85.12} \\
    \midrule
    Overload Mitigation & 33.92 & 33.68 & \textbf{77.08}

## Performance Test

In [16]:
scenario_subset = ["14B_PERF", "30B_PERF"]

print(make_scenario_comparison_table(
    results_perf,
    experiment="performance test",
    scenario_names=scenario_subset,
    label="tab:exp_overview_standard",
))

print(make_scenario_comparison_table(
    results_perf,
    experiment="performance test with 14-bus grid",
    scenario_names=[scenario_subset[0]],
    label="tab:exp_overview_standard",
))

print(make_scenario_comparison_table(
    results_perf,
    experiment="performance test with 30-bus grid",
    scenario_names=[scenario_subset[1]],
    label="tab:exp_overview_standard",
))

\agentresulttable
  {Comparison for the performance test experiment}
  {tab:exp_overview_standard}
  {
    Average Score & \textbf{99.01} & 65.16 \\
    \midrule
    Computational Performance & \textbf{99.01} & 65.16 \\
  }
\agentresulttable
  {Comparison for the performance test with 14-bus grid experiment}
  {tab:exp_overview_standard}
  {
    Average Score & \textbf{98.68} & 63.65 \\
    \midrule
    Computational Performance & \textbf{98.68} & 63.65 \\
  }
\agentresulttable
  {Comparison for the performance test with 30-bus grid experiment}
  {tab:exp_overview_standard}
  {
    Average Score & \textbf{99.34} & 66.67 \\
    \midrule
    Computational Performance & \textbf{99.34} & 66.67 \\
  }
